# 5. Architektura Transformer

## 1. Dane wejściowe

In [1]:
import torch

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)

batch_size = 2
n = 3
d_model = 4

X = torch.randn(batch_size, n, d_model)

print("Seed:", SEED)
print("Kształt X:", tuple(X.shape))
print("X:\n", X)
print("Pierwsza sekwencja:\n", X[0])
print("Pierwszy token pierwszej sekwencji:", X[0, 0])

Seed: 42
Kształt X: (2, 3, 4)
X:
 tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047],
         [ 0.3559, -0.6866, -0.4934,  0.2415]],

        [[-1.1109,  0.0915, -2.3169, -0.2168],
         [-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])
Pierwsza sekwencja:
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [ 0.3559, -0.6866, -0.4934,  0.2415]])
Pierwszy token pierwszej sekwencji: tensor([ 1.9269,  1.4873,  0.9007, -2.1055])


### 1.1 Projekcje Q, K, V

In [2]:
d_k = 2
d_v = 3

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_v)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Kształt W_Q:", tuple(W_Q.shape))
print("Kształt W_K:", tuple(W_K.shape))
print("Kształt W_V:", tuple(W_V.shape))
print("Kształt Q:", tuple(Q.shape))
print("Kształt K:", tuple(K.shape))
print("Kształt V:", tuple(V.shape))
print("W_Q:\n", W_Q)
print("W_K:\n", W_K)
print("W_V:\n", W_V)
print("Pierwszy token x:", X[0, 0])
print("Jego zapytanie q:", Q[0, 0])
print("Jego klucz k:", K[0, 0])
print("Jego wartość v:", V[0, 0])

Kształt W_Q: (4, 2)
Kształt W_K: (4, 2)
Kształt W_V: (4, 3)
Kształt Q: (2, 3, 2)
Kształt K: (2, 3, 2)
Kształt V: (2, 3, 3)
W_Q:
 tensor([[ 1.3525,  0.6863],
        [-0.3278,  0.7950],
        [ 0.2815,  0.0562],
        [ 0.5227, -0.2384]])
W_K:
 tensor([[-0.0499,  0.5263],
        [-0.0085,  0.7291],
        [ 0.1331,  0.8640],
        [-1.0157, -0.8887]])
W_V:
 tensor([[ 0.1498, -0.2089, -0.3870],
        [ 0.9912,  0.4679, -0.2049],
        [-0.7409,  0.3618,  1.9199],
        [-0.2254, -0.3417,  0.3040]])
Pierwszy token x: tensor([ 1.9269,  1.4873,  0.9007, -2.1055])
Jego zapytanie q: tensor([1.2718, 3.0573])
Jego klucz k: tensor([2.1496, 4.7480])
Jego wartość v: tensor([1.5701, 1.3388, 0.0386])


## 2. Dopasowania zapytań i kluczy

In [3]:
K_transposed = K.transpose(-2, -1)
dot_products = Q @ K_transposed
scores = dot_products / (d_k ** 0.5)

print("Kształt K po transpozycji:", tuple(K_transposed.shape))
print("Kształt dopasowań:", tuple(scores.shape))
print("Iloczyny skalarne dla pierwszej sekwencji:\n", dot_products[0])
print("Skalowane dopasowania dla pierwszej sekwencji:\n", scores[0])
print("Dopasowania zapytania pierwszego tokenu do trzech kluczy:", scores[0, 0])
print("Iloczyn q_0 i k_0 policzony osobno:", (Q[0, 0] * K[0, 0]).sum())

Kształt K po transpozycji: (2, 2, 3)
Kształt dopasowań: (2, 3, 3)
Iloczyny skalarne dla pierwszej sekwencji:
 tensor([[17.2498,  4.6220, -3.3278],
        [ 0.3686,  0.6396, -0.0227],
        [-0.3457,  0.7831,  0.1452]])
Skalowane dopasowania dla pierwszej sekwencji:
 tensor([[12.1975,  3.2683, -2.3531],
        [ 0.2607,  0.4523, -0.0160],
        [-0.2444,  0.5538,  0.1026]])
Dopasowania zapytania pierwszego tokenu do trzech kluczy: tensor([12.1975,  3.2683, -2.3531])
Iloczyn q_0 i k_0 policzony osobno: tensor(17.2498)


### 2.1 Wagi atencji - softmax

In [4]:
weights = torch.softmax(scores, dim=-1)

print("Kształt wag:", tuple(weights.shape))
print("Wagi dla pierwszej sekwencji:\n", weights[0])
print("Wagi dla zapytania pierwszego tokenu:", weights[0, 0])
print("Sumy wag po kluczach:\n", weights.sum(dim=-1))
print("Wagi pierwszego zapytania w procentach:", weights[0, 0] * 100)

Kształt wag: (2, 3, 3)
Wagi dla pierwszej sekwencji:
 tensor([[0.9999, 0.0001, 0.0000],
        [0.3368, 0.4079, 0.2554],
        [0.2157, 0.4791, 0.3052]])
Wagi dla zapytania pierwszego tokenu: tensor([0.9999, 0.0001, 0.0000])
Sumy wag po kluczach:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
Wagi pierwszego zapytania w procentach: tensor([99.9867,  0.0132,  0.0000])


### 2.2 Mieszanie wartości V

In [5]:
output = weights @ V
first_contributions = weights[0, 0].unsqueeze(-1) * V[0]

print("Kształt wyjścia:", tuple(output.shape))
print("Wartości V pierwszej sekwencji:\n", V[0])
print("Wagi pierwszego zapytania:", weights[0, 0])
print("Wkłady trzech wartości do wyniku pierwszego zapytania:\n", first_contributions)
print("Suma wkładów:", first_contributions.sum(dim=0))
print("Wyjście pierwszego zapytania:", output[0, 0])
print("Wyjście całej pierwszej sekwencji:\n", output[0])

Kształt wyjścia: (2, 3, 3)
Wartości V pierwszej sekwencji:
 tensor([[ 1.5701,  1.3388,  0.0386],
        [-0.7285, -0.1866, -0.5801],
        [-0.3162, -0.6566, -0.8708]])
Wagi pierwszego zapytania: tensor([0.9999, 0.0001, 0.0000])
Wkłady trzech wartości do wyniku pierwszego zapytania:
 tensor([[ 1.5699,  1.3386,  0.0386],
        [-0.0001, -0.0000, -0.0001],
        [-0.0000, -0.0000, -0.0000]])
Suma wkładów: tensor([1.5698, 1.3385, 0.0385])
Wyjście pierwszego zapytania: tensor([1.5698, 1.3385, 0.0385])
Wyjście całej pierwszej sekwencji:
 tensor([[ 1.5698,  1.3385,  0.0385],
        [ 0.1508,  0.2070, -0.4460],
        [-0.1069, -0.0011, -0.5354]])


### 2.3 Funkcja pojedynczej głowy attention

In [6]:
def attention(Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
    if mask is not None:
        scores = scores + mask
    if torch.isneginf(scores).all(dim=-1).any():
        raise ValueError("Każde zapytanie musi mieć przynajmniej jeden dozwolony klucz.")
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


function_output, function_weights = attention(Q, K, V)

print("Kształt wyjścia funkcji:", tuple(function_output.shape))
print("Kształt wag funkcji:", tuple(function_weights.shape))
print("Wyjście zgodne z wcześniejszym rachunkiem:", torch.allclose(function_output, output))
print("Wagi zgodne z wcześniejszym rachunkiem:", torch.allclose(function_weights, weights))

Kształt wyjścia funkcji: (2, 3, 3)
Kształt wag funkcji: (2, 3, 3)
Wyjście zgodne z wcześniejszym rachunkiem: True
Wagi zgodne z wcześniejszym rachunkiem: True


### 2.4 Maska przyczynowa

In [7]:
causal_mask = torch.triu(
    torch.full((n, n), float("-inf"), dtype=Q.dtype, device=Q.device),
    diagonal=1,
)
causal_output, causal_weights = attention(Q, K, V, mask=causal_mask)

print("Maska przyczynowa:\n", causal_mask)
print("Wagi bez maski dla pierwszej sekwencji:\n", function_weights[0])
print("Wagi z maską dla pierwszej sekwencji:\n", causal_weights[0])
print("Liczba niezerowych wag ponad przekątną:", torch.count_nonzero(torch.triu(causal_weights, diagonal=1)).item())
print("Sumy wag po kluczach:\n", causal_weights.sum(dim=-1))
print("Pierwsze wyjście:", causal_output[0, 0])
print("Pierwsza wartość V:", V[0, 0])
print("Pierwsze wyjście równe pierwszej wartości:", torch.allclose(causal_output[:, 0], V[:, 0]))

Maska przyczynowa:
 tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])
Wagi bez maski dla pierwszej sekwencji:
 tensor([[0.9999, 0.0001, 0.0000],
        [0.3368, 0.4079, 0.2554],
        [0.2157, 0.4791, 0.3052]])
Wagi z maską dla pierwszej sekwencji:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.4522, 0.5478, 0.0000],
        [0.2157, 0.4791, 0.3052]])
Liczba niezerowych wag ponad przekątną: 0
Sumy wag po kluczach:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
Pierwsze wyjście: tensor([1.5701, 1.3388, 0.0386])
Pierwsza wartość V: tensor([1.5701, 1.3388, 0.0386])
Pierwsze wyjście równe pierwszej wartości: True


## Wykorzystanie AI

- Narzędzie: OpenCode (OpenChamber), model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol).
- Zakres: wcześniejsze pełne rozwiązanie z wynikami zastąpiono początkiem zadania do wspólnej pracy. AI przygotowało tensor wejściowy, projekcje Q, K, V, rachunek attention, funkcję z opcjonalną maską, maskę przyczynową i objaśnienia w rozmowie. Student uruchamiał kolejne komórki. Na jego wyraźną prośbę AI sprawdziło notebook i wykonało wszystkie siedem komórek kodu od początku w świeżym kernelu Pythona, bez błędów, zapisując wyniki.
- Historia promptów znajduje się w sekcji poniżej oraz w [used_prompts.md](used_prompts.md).

## Used prompts

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
/zajecia-nlp https://christop.faculty.wmi.amu.edu.pl/index.php?page=chapter&id=uczenie-glebokie-w-przetwarzaniu-tekstu&topic=architektura-transformer
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
zmien skill aby nie uruchamial automatycznie testow, ma polegac on na wspolpracy miedzy studentem a AI i celem jest zrozumienie zagadnienia
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok wiec zresetuj teraz i dawaj
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Odpowiedź na pytanie o zakres resetu: Notebook 5 od początku
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
nie mam pojcia, ale uruchomilem komorke
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
2x [x, x, x], gdzie x = [c, c, c, c]?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok, co dalej?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
pierwszy?
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Kształt K po transpozycji: (2, 2, 3)
Kształt dopasowań: (2, 3, 3)
Iloczyny skalarne dla pierwszej sekwencji:
 tensor([[17.2498,  4.6220, -3.3278],
        [ 0.3686,  0.6396, -0.0227],
        [-0.3457,  0.7831,  0.1452]])
Skalowane dopasowania dla pierwszej sekwencji:
 tensor([[12.1975,  3.2683, -2.3531],
        [ 0.2607,  0.4523, -0.0160],
        [-0.2444,  0.5538,  0.1026]])
Dopasowania zapytania pierwszego tokenu do trzech kluczy: tensor([12.1975,  3.2683, -2.3531])
Iloczyn q_0 i k_0 policzony osobno: tensor(17.2498)
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
Ok wygląda prawidłowego
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
ok
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
oki
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
dodaj used prompts do aktualnego pliku na koncu
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
oraz sprawdz notebook
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
chdozilo mi pythonem
```

```text
tool: OpenCode (OpenChamber)
model: GPT-6.1 Sol (github-copilot/gpt-6.1-sol)
w skillu jest tscrpt
```
